In [1]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing Libraries
from dotenv import load_dotenv
from anthropic import Anthropic

In [3]:
# Load env variables
load_dotenv()

True

In [4]:
# Create an API client
client = Anthropic()
model = "claude-sonnet-4-0"

In [5]:
# Make a request
resp = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "What is cloud computing? Answer in a sentence."
        }
    ]
)

In [6]:
resp

Message(id='msg_01XZqmUDGxT9VGxtbaAYKrPy', container=None, content=[TextBlock(citations=None, text='Cloud computing is the delivery of computing services—including servers, storage, databases, networking, software, and analytics—over the internet ("the cloud") to offer faster innovation, flexible resources, and economies of scale.', type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=17, output_tokens=46, server_tool_use=None, service_tier='standard'))

In [7]:
resp.content[0].text

'Cloud computing is the delivery of computing services—including servers, storage, databases, networking, software, and analytics—over the internet ("the cloud") to offer faster innovation, flexible resources, and economies of scale.'

### The Problem with Stateless Conversations

In [9]:
# Make another request
resp2 = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "Write another sentence."
        }
    ]
)

In [10]:
resp2.content[0].text

'The old lighthouse stood silently on the rocky cliff, its beacon long extinguished but its weathered stone walls still standing guard over the restless sea below.'

> Claude "What is cloud computing?" and get a good response. Then you follow up with "Write another sentence" - Claude has no idea what you're referring to. It will write a sentence about something completely random because it has no memory of the cloud computing discussion.

### How Multi-Turn Conversations Work
To maintain conversation context, you need to do two things:
- Manually maintain a list of all messages in your code
- Send the complete message history with every request

![image.png](https://everpath-course-content.s3-accelerate.amazonaws.com/instructor%2Fa46l9irobhg0f5webscixp0bs%2Fpublic%2F1748623270%2F03_-_004_-_Multi-Turn_Conversations_02.1748623270625.png)

Here's the flow that actually works:
- Send your initial user message to Claude
- Take Claude's response and add it to your message list as an assistant message
- Add your follow-up question as another user message
- Send the entire conversation history to Claude

![image.png](https://everpath-course-content.s3-accelerate.amazonaws.com/instructor%2Fa46l9irobhg0f5webscixp0bs%2Fpublic%2F1748623271%2F03_-_004_-_Multi-Turn_Conversations_08.1748623271832.png)

In [11]:
def add_user_message(messages, text):
    """
    Helper function to add a user message to the conversation history.
    Args:        
        messages (list): The conversation history, a list of message dicts.
        text (str): The content of the user's message.
    """
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    """
    Helper function to add an assistant message to the conversation history.
    Args:
        messages (list): The conversation history, a list of message dicts.
        text (str): The content of the assistant's message.
    """
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    """Anthropic API chat function that takes a list of messages and returns the assistant's response text."""
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

In [14]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define cloud computing in one sentence")

# Get Claude's response
answer = chat(messages)
print("Claude's response:", answer)

Claude's response: Cloud computing is the delivery of computing services—including servers, storage, databases, networking, software, and analytics—over the internet ("the cloud") to offer faster innovation, flexible resources, and economies of scale.


In [15]:
# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)
print("Claude's follow-up response:", final_answer)

Claude's follow-up response: Users can access these cloud-based resources on-demand from anywhere with an internet connection, paying only for what they use rather than investing in expensive on-premises hardware and infrastructure.
